# 第二轮20题原始化学审计

分层抽样、不与第一轮重复；无Poe调用。AI语义审读不是人类金标准。用本目录及安装RDKit的Python运行；保存输出来自普通Python单元顺序执行器，不是Jupyter内核。

In [1]:
import json
from audit_round2 import HERE, run
from verify_round2 import verify
evidence = run()


{
  "origins": 20,
  "overlap_with_round1": 0,
  "steps": 106,
  "checks": 246,
  "failed_checks": [],
  "reconstructed": 20,
  "selected": [
    "mol_edit.add_v2.0013",
    "mol_edit.add_v2.0035",
    "mol_edit.add_v2.0111",
    "mol_edit.add_v2.0119",
    "mol_edit.add_v2.0172",
    "mol_edit.add_v2.0183",
    "mol_edit.add_v2.0229",
    "mol_edit.delete_v2.0016",
    "mol_edit.delete_v2.0036",
    "mol_edit.delete_v2.0108",
    "mol_edit.delete_v2.0134",
    "mol_edit.delete_v2.0273",
    "mol_edit.delete_v2.0275",
    "mol_edit.delete_v2.0299",
    "mol_edit.substitute_v2.0032",
    "mol_edit.substitute_v2.0057",
    "mol_edit.substitute_v2.0064",
    "mol_edit.substitute_v2.0122",
    "mol_edit.substitute_v2.0150",
    "mol_edit.substitute_v2.0281"
  ]
}


In [2]:
verification = verify()


{
  "origins": 20,
  "steps": 106,
  "overlap_with_round1": 0,
  "numeric_structure_checks_passed": 246,
  "exact_graph_reconstructions": 20,
  "verbatim_finding_quotes_verified": 5,
  "source_hashes_unchanged": 6,
  "previous_audit_dependencies_unchanged": true,
  "deliberate_error_detection_tests_passed": 3,
  "review_categories": {
    "no_material_issue_found": 15,
    "factual_error": 4,
    "underspecified_stereochemistry": 1
  },
  "metadata_scope_warnings": 8,
  "live_poe_requests": 0
}


## 全部20题：输入、计算结果与逐题审读

原数据all_pass不作为真值证明；语义结论来自原始题目、结构和完整106步的AI审读。

In [3]:
review = json.loads((HERE / 'review.json').read_text())
by_id = {r['origin_id']:r for r in review['records']}
for r in evidence['records']:
    print(r['origin_id'], r['raw']['instruction'])
    print('SOURCE', r['raw']['indexed_smiles'])
    print('GT', r['raw']['gt_smiles'])
    print('COUNTS', r['source_computed'], r['answer_computed'])
    print('REVIEW', by_id[r['origin_id']])


mol_edit.add_v2.0013 Please acetylate the secondary aliphatic amine.
SOURCE [CH3:1][NH:2][CH2:3][C@H:4]([CH3:5])[O:6][c:7]1[cH:8][cH:9][cH:10][c:11]2[n:12][cH:13][n:14][c:15]([NH:16][c:17]3[cH:18][cH:19][c:20]([O:21][c:22]4[cH:23][cH:24][c:25]([CH3:26])[n:27][cH:28]4)[c:29]([CH3:30])[cH:31]3)[c:32]12
GT CC(=O)N(C)C[C@H](C)Oc1cccc2ncnc(Nc3ccc(Oc4ccc(C)nc4)c(C)c3)c12
COUNTS {'heavy_atoms': 32, 'rings': 4, 'formula': 'C25H27N5O2', 'elements': {'C': 25, 'N': 5, 'O': 2}, 'formal_charge': 0} {'heavy_atoms': 35, 'rings': 4, 'formula': 'C27H29N5O3', 'elements': {'C': 27, 'O': 3, 'N': 5}, 'formal_charge': 0}
REVIEW {'origin_id': 'mol_edit.add_v2.0013', 'category': 'no_material_issue_found', 'review': 'N2是目标仲脂肪胺而非N16芳胺；乙酰基含2C+O，通过羰基C成键，+3重原子。喹唑啉2环、苯1环、吡啶1环正确；已有立体标记保留。'}
mol_edit.add_v2.0035 Please add a sulfamoyl group to the primary alcohol.
SOURCE [O:1]=[C:2]([NH:3][C@@H:4]1[CH2:5][C@H:6]([CH2:7][OH:8])[C@@H:9]([O:10][C:11](=[O:12])[c:13]2[cH:14][cH:15][cH:16][cH:17][cH:18]2)[C@H:19]1[O:20][C:

## 具体发现与结构反例

计数正确≠骨架认对；GT可重建≠题目唯一确定立体构型。

In [4]:
print(json.dumps(review['findings'], ensure_ascii=False, indent=2))
print((HERE / 'semantic_checks.json').read_text())
print(json.dumps(review['metadata_notes'], ensure_ascii=False, indent=2))


[
  {
    "id": "R2-F1",
    "origin_id": "mol_edit.add_v2.0111",
    "steps": [
      5
    ],
    "category": "factual_error",
    "severity": "medium; can contaminate negative labels",
    "confidence": "high",
    "quote": "The source contains 6 rings (phenyl, pyridine, thieno[3,2-c]pyridine (2 rings), thiazole, and piperidine).",
    "explanation": "thieno[3,2-c]pyridine应为thieno[3,2-b]pyridine。该稠环包含map10–18，吡啶N13直接邻接稠合C14；PubChem的b型结构匹配这些原子，c型结构不匹配。两种骨架同分子式、同环数，故重原子数和环数校验无法区分。",
    "recommendation": "修正Step5骨架名称；保留N30乙酰化产物与36→39重原子、6→6环的计数。",
    "sources": [
      {
        "url": "https://pubchem.ncbi.nlm.nih.gov/compound/Thienopyridine",
        "label": "PubChem CID 12210218: thieno[3,2-b]pyridine",
        "smiles": "C1=CC2=C(C=CS2)N=C1"
      },
      {
        "url": "https://pubchem.ncbi.nlm.nih.gov/compound/Thieno_3_2-c_pyridine",
        "label": "PubChem CID 67500: thieno[3,2-c]pyridine",
        "smiles": "C1=CN=CC2=C1SC=C2"
      }
    ]
  },
  {
    "id": "R2-F2",
 

## 限制

不估计上游总体错误率，不验证实验收率或反应选择性；对rxn_cls仅报告与本题编辑不符，继承字段的真实定义尚未确认。完整建议与来源链接见REPORT.md和review.json。